# Relevance Feedback and Query Expansion: Rocchio and RM3

This notebook is the *narrative* pillar: it imports the canonical reference implementation in `pseudo_relevance_feedback.py` — which **owns every number** — and walks the topic section by section. Each claim is an `assert` in the `.py`; here we run those checks and print the numbers the page and the interactive lab mirror.

Run end to end with:

```bash
uv run --with numpy --with jupyter \
    jupyter execute notebooks/pseudo-relevance-feedback/01_pseudo_relevance_feedback.ipynb
```

In [ ]:
import sys, pathlib
_cands = [pathlib.Path.cwd(),
          pathlib.Path.cwd() / "notebooks" / "pseudo-relevance-feedback",
          pathlib.Path("notebooks/pseudo-relevance-feedback")]
for _d in _cands:
    if (_d / "pseudo_relevance_feedback.py").exists():
        sys.path.insert(0, str(_d)); break
import pseudo_relevance_feedback as prf
index = prf.Index(prf._CORPUS)
print("loaded reference implementation; query =", prf._QUERY, " relevant =", sorted(prf._RELEVANT))

## 1. The vocabulary-mismatch problem

The query *rate guidance* should retrieve four relevant documents, but two of them speak only of the *outlook* and *forecast* for rates and never use the word "guidance". Lexical retrieval cannot match what is not there, so the baseline ranking buries those synonym documents beneath non-relevant ones that happen to contain "guidance". Relevance feedback fixes this by **expanding** the query with terms harvested from the documents already retrieved.

In [ ]:
prf.recall_curve()

## 2. Rocchio: moving the query toward the relevant centroid

The vector-space answer is **Rocchio**: $q' = \alpha\,q + \beta\,\frac{1}{|D_r|}\sum_{d\in D_r} d - \gamma\,\frac{1}{|D_{nr}|}\sum_{d\in D_{nr}} d$. With pseudo-relevance feedback ($\gamma=0$) the query is pulled toward the centroid of the top retrieved documents, which raises its similarity to them — and to their unseen synonym neighbors.

In [ ]:
prf.test_rocchio_moves_to_centroid_and_helps()

## 3. The relevance model RM1

The language-model answer estimates a **relevance model** $P(w\mid R) = \sum_d P(w\mid d)\,P(d\mid q)$, weighting each feedback document by its query likelihood $P(d\mid q)\propto P(q\mid d)$ (the [query-likelihood](/topics/query-likelihood-language-models) score). It is a proper distribution over the vocabulary, and its top terms are exactly the expansion terms — *forecast* and *outlook* dominate when the feedback is clean.

In [ ]:
prf.test_rm1_is_distribution()
prf.expansion_terms()

## 4. RM3: interpolating with the original query

RM3 mixes the relevance model back with the original query, $P_{\text{RM3}}(w) = (1-\alpha)P_{\text{ml}}(w\mid q) + \alpha\,P_{\text{RM1}}(w\mid R)$, and re-scores documents by the cross-entropy of this expanded model — the same KL view the query-likelihood topic established. At $\alpha=0$ it is the original query; a little feedback bridges the vocabulary gap and lifts recall to 1.0.

In [ ]:
prf.test_rm3_improves_recall()
prf.test_rm3_alpha_limits()

## 5. Query drift

Pseudo-relevance feedback assumes the top documents are relevant. When they are not — when the feedback set grows to include off-topic documents — their terms enter the relevance model, the query drifts toward them, and a genuinely relevant document is demoted. Recall rises with a little feedback and falls with too much; there is no per-query guarantee.

In [ ]:
prf.test_rm3_query_drift()

## 6. The numbers the viz mirrors

`QueryExpansionFeedbackLab.tsx` indexes a precomputed table by the feedback slider: per feedback size, the RM3 ranking, the recall, and the top RM1 expansion terms with weights — so the page, the notebook, and the lab share one set of numbers.

In [ ]:
prf.viz_constants()

---

**Three pillars, one set of numbers.** The rigorous math (the page), the interactive feedback laboratory, and this notebook all read from `pseudo_relevance_feedback.py`. Change a number there and the viz and the page must change with it.